# The driven damped pendulum (Taylor Chapter 12)

*Class notebook for Chapter 12, and the program for HW13 Problems 12.6, 12.7 (and the optional 12.8-12.10).* Run each cell in order with **Shift-Enter**. You only need to edit the cells marked **Your turn**.

A pendulum with damping, pushed by a sinusoidal drive, with $\sin\phi$ left as $\sin\phi$, obeys Taylor's (12.11):
$$\ddot\phi + 2\beta\dot\phi + \omega_0^2\sin\phi = \gamma\,\omega_0^2\cos(\omega t).$$
The drive strength $\gamma = F_0/(mg)$ compares the drive with the weight. Throughout Chapter 12 Taylor uses
$$\omega = 2\pi \text{ rad/s (so the drive period is 1 s)}, \qquad \omega_0 = 1.5\,\omega, \qquad \beta = \omega_0/4 ,$$
and so does this notebook. Every time axis is in seconds, which here is the same as counting drive periods.

There is no formula for $\phi(t)$, so we solve (12.11) numerically and look at the answers.

## 1. Predict first (before you run anything)

Double-click the cell below this one, type your answers, then Shift-Enter. A sentence each is plenty.

- With a drive weaker than the weight ($\gamma < 1$), start the pendulum from three different angles. After a few drive periods, are the three motions different or the same?
- Once the start-up wiggles have died away, does the motion always repeat with the drive's period, 1 s? If not, what else could it do?
- Suppose you look at the pendulum only once per drive period, at $t = 0, 1, 2, \ldots$ s, and mark a dot each time. How many different dots do you see if the motion repeats every 2 drive periods? If it never repeats?

*(your prediction here)*

**Three more, for sections 11-13** (answer these before you run anything in those sections):

- (a) Plot the strobed angle (one dot per drive period, after the start-up has died away) above the value of $\gamma$ on the horizontal axis. What does that column of dots look like for a motion of period 4? For a motion that never repeats?
- (b) Two pendulums at $\gamma = 1.084$, released from rest $10^{-5}$ rad apart. About how many drive periods until they are a whole radian apart: 1, 7, 100, never?
- (c) Start a small square of states in the $(\phi, \dot\phi)$ plane at $\gamma = 1.105$ and let every point in it move for one drive period. Is the patch's area bigger, smaller, or the same? By roughly what factor?

*(your prediction here)*

## 2. Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact

omega = 2 * np.pi        # rad/s, drive frequency (drive period 1 s)
omega0 = 1.5 * omega     # rad/s, natural frequency
beta = omega0 / 4        # 1/s, damping constant
samples_per_second = 200

## 3. The solver

`solve_ddp` ("driven damped pendulum") is the one function everything else uses. You give it a drive strength `gamma`, a starting angle `phi0` (radians), and how long to run `t_max` (seconds); it starts the pendulum at rest unless you also give `phidot0`.

In [ ]:
def solve_ddp(gamma, phi0, t_max, phidot0=0.0):
    """Solve (12.11) from phi(0) = phi0, phidot(0) = phidot0.
    Returns times t (s), angles phi (rad) and angular velocities phidot (rad/s)."""

    def rates(t, state):
        phi, phidot = state
        phiddot = (-2 * beta * phidot - omega0**2 * np.sin(phi)
                   + gamma * omega0**2 * np.cos(omega * t))
        return [phidot, phiddot]

    t = np.linspace(0, t_max, int(samples_per_second * t_max) + 1)
    sol = solve_ivp(rates, (0, t_max), [phi0, phidot0], t_eval=t,
                    rtol=1e-10, atol=1e-10)
    return sol.t, sol.y[0], sol.y[1]

Line by line:

- The solver wants a second-order equation as two first-order ones, so the "state" is the pair (angle, angular velocity), just as in the HW01 skateboard notebook. `rates` says how fast each one changes: the angle changes at `phidot`, and `phidot` changes at the $\ddot\phi$ that (12.11) gives.
- `np.linspace(...)` asks for the answer 200 times per second, so that every whole second ($t = 0, 1, 2, \ldots$, once per drive period) is one of the sample times. Section 6 uses that.
- `rtol` and `atol` set how accurate the solver must be. They are tight on purpose: in Chapter 12 small numerical errors can grow, so we pay for accuracy.
- The function hands back three arrays; `t, phi, phidot = solve_ddp(...)` catches all three.

## 4. A first run

A weak drive, $\gamma = 0.2$, from rest at $\phi = 0$ (Taylor's Figure 12.2):

In [ ]:
t, phi, phidot = solve_ddp(0.2, 0.0, 6)

plt.figure(figsize=(8, 3.5))
plt.plot(t, phi)
plt.xlabel("t (seconds)")
plt.ylabel("phi (radians)")
plt.title("gamma = 0.2, phi(0) = 0")
plt.grid(alpha=0.3)
plt.show()

After about one drive period the start-up wiggle is gone, and what is left repeats every 1 s, like the linear driven oscillator of Chapter 5.

## 5. Same place, different number: wrapping $\phi$

The angles $\phi$ and $\phi + 2\pi$ are the same position of the pendulum. The solver doesn't know that: if the pendulum goes over the top, $\phi$ just keeps counting. Here is a strong drive, $\gamma = 1.4$, which sends the pendulum over the top once every drive period (Taylor's Figure 12.19). Its $\phi$ climbs (or falls) without end:

In [ ]:
def wrap(phi):
    """The same angle, moved into -pi < phi <= pi by adding or subtracting whole turns (2 pi)."""
    return np.pi - np.mod(np.pi - phi, 2 * np.pi)


print(wrap(7.0), "which is 7 - 2 pi =", 7.0 - 2 * np.pi)
print(wrap(-np.pi), "and", wrap(np.pi), "(both pi: -pi and pi are the same place)")

`np.mod(x, 2 * np.pi)` is the remainder after removing whole multiples of $2\pi$, a number between 0 and $2\pi$; the two `np.pi -` around it shift that range to $(-\pi, \pi]$.

`plot_runs` puts both pictures side by side for one or more starting angles: on the left $\phi$ as the solver computed it, on the right the same runs wrapped. The wrapped curves are drawn as dots, because a line would draw a vertical stroke each time the angle jumps from $\pi$ to $-\pi$. (So a swing whose crest reaches just past $\pi$ shows a few dots along the bottom edge: the same place, under its other label.)

In [ ]:
def plot_runs(gamma, start_angles, t_max):
    """phi(t) for each starting angle (released from rest): as computed, and wrapped."""
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    for phi0 in start_angles:
        t, phi, phidot = solve_ddp(gamma, phi0, t_max)
        left.plot(t, phi, label=f"phi(0) = {phi0:.3f}")
        right.plot(t, wrap(phi), ".", markersize=1.5)
    left.set_title(f"gamma = {gamma}: phi as computed")
    right.set_title("the same runs, wrapped into (-pi, pi]")
    right.set_ylim(-np.pi, np.pi)
    for ax in (left, right):
        ax.set_xlabel("t (seconds)")
        ax.set_ylabel("phi (radians)")
        ax.grid(alpha=0.3)
    left.legend()
    plt.show()


plot_runs(1.4, [0.0], 8)

On the left, $\phi$ drops by about $2\pi$ every second: the pendulum goes all the way round once per drive period. On the right the same motion is a pattern that repeats every second.

## 6. Looking once per drive period: the period of the settled motion

To find out how often the settled motion repeats, look at the pendulum once per drive period ("strobe" it): at $t = 0, 1, 2, \ldots$ s, skipping the first `t_settle` seconds while the start-up behavior dies away. If the strobed angle comes back to the same value every $n$ looks (allowing for whole turns), the motion has period $n$ drive periods.

- `strobe` picks every 200th sample (one per second) after `t_settle` and wraps the angles.
- `find_period` tries $n = 1, 2, 3, \ldots$ and returns the first $n$ for which every strobed angle matches the one $n$ looks later, to within `tolerance` radians. It returns `None` if nothing up to `longest` fits: either the motion never repeats, or it has not settled yet.

In [ ]:
def strobe(t, phi, phidot, t_settle):
    """phi (wrapped) and phidot once per drive period, for t >= t_settle."""
    t_once = t[::samples_per_second]
    keep = t_once >= t_settle
    phi_once = phi[::samples_per_second][keep]
    phidot_once = phidot[::samples_per_second][keep]
    return wrap(phi_once), phidot_once


def find_period(phi_strobe, tolerance=1e-3, longest=16):
    """Smallest n with phi(t + n) = phi(t) for every strobe (up to whole turns), or None."""
    for n in range(1, longest + 1):
        shift = wrap(phi_strobe[n:] - phi_strobe[:-n])
        if np.all(np.abs(shift) < tolerance):
            return n
    return None

Pick a drive strength and a starting angle. The left panel is the first 20 s; the right panel is the last 10 s of a 364 s run, with the strobe times marked as dots. Each choice takes a second or two to compute.

These are the drive strengths from class (Taylor Figures 12.5-12.8): try them from both starting angles. Two things to watch for: the period doubling as $\gamma$ creeps up from 1.078 toward 1.083, and $\gamma = 1.077$, where the starting angle decides which settled motion you get.

In [ ]:
gamma_choices = [1.077, 1.078, 1.081, 1.082, 1.0826, 1.105]
start_choices = {"phi(0) = -pi/2": -np.pi / 2, "phi(0) = 0": 0.0}
t_settle = 300


@interact(gamma=gamma_choices, phi0=start_choices)
def explore(gamma=1.078, phi0=-np.pi / 2):
    t, phi, phidot = solve_ddp(gamma, phi0, t_settle + 64)
    phi_s, phidot_s = strobe(t, phi, phidot, t_settle)
    n = find_period(phi_s)

    fig, (early, late) = plt.subplots(1, 2, figsize=(11, 3.8))
    first = t <= 20
    early.plot(t[first], phi[first])
    early.set_title(f"gamma = {gamma}: the first 20 s")
    last = t >= t[-1] - 10
    late.plot(t[last], wrap(phi[last]), ".", markersize=1.5)
    late.plot(np.arange(t[-1] - 10, t[-1] + 0.5), phi_s[-11:], "ro", label="strobe")
    late.set_title("the last 10 s, wrapped")
    late.legend(loc="upper right")
    for ax in (early, late):
        ax.set_xlabel("t (seconds)")
        ax.set_ylabel("phi (radians)")
        ax.grid(alpha=0.3)
    plt.show()
    print("last 8 strobed angles (rad):", np.round(phi_s[-8:], 4))
    print("period (drive periods):", n)

## 7. The Poincare section

Now plot each strobed state as a dot in the $(\phi, \dot\phi)$ plane: one dot per drive period, hundreds of drive periods (a Poincare section, Taylor 12.8). A motion with period $n$ keeps landing on the same $n$ dots. A motion that never repeats gives dots that never land on each other, but they do not fill the plane either. With 600 dots this takes a few seconds.

In [ ]:
@interact(gamma=gamma_choices)
def poincare(gamma=1.082):
    t, phi, phidot = solve_ddp(gamma, -np.pi / 2, t_settle + 600)
    phi_s, phidot_s = strobe(t, phi, phidot, t_settle)

    plt.figure(figsize=(6, 5))
    plt.plot(phi_s, phidot_s, ".", markersize=3)
    plt.xlabel("phi (radians, wrapped)")
    plt.ylabel("phidot (rad/s)")
    plt.title(f"gamma = {gamma}, phi(0) = -pi/2: one dot per drive period")
    plt.grid(alpha=0.3)
    plt.show()

## 8. Your turn: Problems 12.6 and 12.7

`plot_runs` (section 5) does everything these two problems need. Set the drive strength, the list of starting angles (radians, all released from rest), and the run time from the problem; then run the cell. For 12.7, change the three lines and run it again (or copy the cell).

In [ ]:
# Change the line below.
gamma = 0.2
# Change the line below.
start_angles = [0.0]
# Change the line below.
t_max = 6.0

plot_runs(gamma, start_angles, t_max)

## 9. Your turn (optional): Problems 12.8-12.10

`plot_early_late` solves for $0 < t < 50$ s and plots two windows: the first `t_early` seconds, and $40 < t < 50$ s, when the start-up behavior is long gone. It also prints the wrapped strobe angles at $t = 40, 41, \ldots, 50$ s, so you can read the period off the numbers as well as the picture (or call `find_period` on them, section 6).

In [ ]:
def plot_early_late(gamma, phi0, t_early):
    t, phi, phidot = solve_ddp(gamma, phi0, 50)
    fig, (early, late) = plt.subplots(1, 2, figsize=(11, 3.5))
    first = t <= t_early
    early.plot(t[first], phi[first])
    early.set_title(f"gamma = {gamma}, phi(0) = {phi0:.3f}: 0 < t < {t_early} s")
    last = t >= 40
    late.plot(t[last], phi[last])
    late.set_title("40 < t < 50 s")
    for ax in (early, late):
        ax.set_xlabel("t (seconds)")
        ax.set_ylabel("phi (radians)")
        ax.grid(alpha=0.3)
    plt.show()
    phi_s, phidot_s = strobe(t, phi, phidot, 40)
    print("wrapped phi at t = 40, 41, ..., 50 s:", np.round(phi_s, 4))

In [ ]:
# Change the line below.
gamma = 0.2
# Change the line below.
phi0 = 0.0
# Change the line below.
t_early = 10

plot_early_late(gamma, phi0, t_early)

For 12.10, `plot_runs` with `gamma = 1.073`, `t_max = 50` and a list of starting angles between $-\pi$ and $\pi$ puts many starts on one picture; `strobe` and `find_period` (section 6) give the numbers.

## 10. What you found

Double-click each answer cell, type, Shift-Enter. Put the plots (screenshots are fine) and these answers in your HW13 PDF.

**a. 12.6.** Do your three runs bear out the claim that all solutions approach the same periodic attractor? After about how many drive periods? Compare with your first prediction.

*(your answer here)*

**b. 12.7.** Is there still a unique attractor at $\gamma = 1.06$? Use both panels of your plot in the answer.

*(your answer here)*

**c. 12.8-12.10 (optional).** What is the period of the long-term motion, and does it depend on where you start?

*(your answer here)*

**d. What didn't work** the first time, if anything.

*(your answer here)*

## 11. The bifurcation diagram: every drive strength in one picture

Section 6 strobed one $\gamma$ at a time. Draw the strobed angles as a column of dots above that $\gamma$, do that for a few hundred values of $\gamma$, and stack the columns side by side: that picture (Taylor 12.6) holds the whole period-doubling cascade at once.

**Predict:** look back at your prediction (a) in section 1. Then run the cell below: one $\gamma$ (Taylor's 1.082, from $\phi(0) = -\pi/2$), one column. It uses `solve_ddp` and `strobe` from sections 3 and 6 and takes a second or two.

In [ ]:
gamma = 1.082
t, phi, phidot = solve_ddp(gamma, -np.pi / 2, t_settle + 60)
phi_s, phidot_s = strobe(t, phi, phidot, t_settle)

plt.figure(figsize=(4, 5))
plt.plot(np.full(len(phi_s), gamma), phi_s, "k.", markersize=4)
plt.xlabel("gamma")
plt.ylabel("phi at t = 300, 301, ..., 360 s (radians, wrapped)")
plt.title(f"gamma = {gamma}: one column")
plt.grid(alpha=0.3)
plt.show()
print("last 8 strobed angles (rad):", np.round(phi_s[-8:], 4))
print("period (drive periods):", find_period(phi_s))

Sixty looks, but they land on only a few places. To draw a few hundred columns, a loop over `solve_ddp` would take minutes. Instead, `step_all` moves *every* pendulum forward together: `phi` and `phidot` are numpy arrays with one entry per pendulum, and each line of the update acts on all of them at once. It is the Runge-Kutta method (RK4), the recipe inside most ODE solvers, written out by hand with a fixed step of 1/100 of a drive period.

- `accel` is the right-hand side of (12.11), the same formula as `rates` in `solve_ddp`.
- `step_all` takes `n_steps` steps of size `dt`, starting at time `t_start`. `gamma` can be one number or an array with one drive strength per pendulum.
- `bifurcation` starts one pendulum per $\gamma$ at rest from $\phi(0) = -\pi/2$, lets all of them run 300 drive periods so the start-up dies away, then records the wrapped $\phi$ at the next 60 whole seconds.

In [ ]:
steps_per_period = 100
dt = 1 / steps_per_period


def accel(t, phi, phidot, gamma):
    """phi-ddot from (12.11)."""
    return (-2 * beta * phidot - omega0**2 * np.sin(phi)
            + gamma * omega0**2 * np.cos(omega * t))


def step_all(gamma, phi, phidot, t_start, n_steps):
    """RK4 for many pendulums at once: phi and phidot are arrays."""
    for k in range(n_steps):
        t = t_start + k * dt
        a1 = accel(t, phi, phidot, gamma)
        v2 = phidot + dt / 2 * a1
        a2 = accel(t + dt / 2, phi + dt / 2 * phidot, v2, gamma)
        v3 = phidot + dt / 2 * a2
        a3 = accel(t + dt / 2, phi + dt / 2 * v2, v3, gamma)
        v4 = phidot + dt * a3
        a4 = accel(t + dt, phi + dt * v3, v4, gamma)
        phi = phi + dt / 6 * (phidot + 2 * v2 + 2 * v3 + v4)
        phidot = phidot + dt / 6 * (a1 + 2 * a2 + 2 * a3 + a4)
    return phi, phidot


def bifurcation(gamma_min, gamma_max, n_gamma=300, show_splits=False):
    """Strobed phi (wrapped) after 300 drive periods, plotted against gamma."""
    gammas = np.linspace(gamma_min, gamma_max, n_gamma)
    phi = np.full(n_gamma, -np.pi / 2)
    phidot = np.zeros(n_gamma)
    phi, phidot = step_all(gammas, phi, phidot, 0, 300 * steps_per_period)

    plt.figure(figsize=(10, 5))
    for n in range(300, 360):
        phi, phidot = step_all(gammas, phi, phidot, n, steps_per_period)
        plt.plot(gammas, wrap(phi), "k.", markersize=1)
    if show_splits:
        for split in [1.0663, 1.0793, 1.0821]:
            plt.axvline(split, color="r", linestyle="--", linewidth=1)
    plt.xlim(gamma_min, gamma_max)
    plt.xlabel("gamma")
    plt.ylabel("phi at t = 301, ..., 360 s (radians, wrapped)")
    plt.title(f"Bifurcation diagram, phi(0) = -pi/2, {n_gamma} values of gamma")
    plt.grid(alpha=0.3)
    plt.show()

**Your turn.** The cell below draws the diagram from 1.060 to 1.088 (about a second). Read off where each column splits in two. Then set `show_splits = True` to draw Taylor's Table 12.1 values as dashed lines.

Then zoom in just past the end of the cascade: change the range to 1.083 to 1.088. Is it chaos all the way across, or is there order inside the chaos? For a single $\gamma$ (1.0845, 1.0846, 1.087, ...), go back to the first cell of this section, change `gamma`, and run it.

In [ ]:
# Change the two lines below.
gamma_min = 1.060
gamma_max = 1.088
# Change to True to mark Table 12.1's splits.
show_splits = False

bifurcation(gamma_min, gamma_max, show_splits=show_splits)

## 12. Two pendulums, $10^{-5}$ rad apart (Problem 12.14)

Run two pendulums with the same $\gamma$ from slightly different starting angles and watch their difference $\Delta\phi(t)$. Taylor's (12.26) says $\Delta\phi(t) \sim K e^{\lambda t}$; the *Lyapunov exponent* $\lambda$ is negative when nearby motions come together and positive when they fly apart.

`separation` plots $\log_{10}|\Delta\phi|$ against $t$ (using $\phi$ as computed, not wrapped), marks the biggest $|\Delta\phi|$ in each drive period (the crests, red dots), and fits a straight line through the crests. On a $\log_{10}$ axis the slope of that line is $\lambda / \ln 10$, so the cell prints both the slope and $\lambda = \text{slope} \times \ln 10$ (Taylor's $\lambda$ uses the natural log).

**Predict:** look back at your prediction (b) in section 1, then run the next two cells. The second one has a menu: 12.14 is the class problem; the others are 12.15, Taylor's Figure 12.13, and the run from the slides.

In [ ]:
def separation(gamma, phi0_a, phi0_b, t_max):
    """log10 of the gap between two runs, its crest each drive period, and a fit."""
    t, phi_a, phidot_a = solve_ddp(gamma, phi0_a, t_max)
    t, phi_b, phidot_b = solve_ddp(gamma, phi0_b, t_max)
    log_gap = np.log10(np.abs(phi_b - phi_a))

    crest_t = []
    crest = []
    for n in range(int(round(t_max))):
        period = (t >= n) & (t < n + 1)
        i = np.argmax(log_gap[period])
        crest_t.append(t[period][i])
        crest.append(log_gap[period][i])
    slope, intercept = np.polyfit(crest_t, crest, 1)

    plt.figure(figsize=(9, 4))
    plt.plot(t, log_gap, linewidth=1)
    plt.plot(crest_t, crest, "ro", label="crest, one per drive period")
    plt.plot(t, slope * t + intercept, "r--", label=f"fit: slope {slope:.2f} per s")
    plt.xlabel("t (seconds)")
    plt.ylabel("log10 |phi_b - phi_a|  (radians)")
    plt.title(f"gamma = {gamma}: phi(0) = {phi0_a:.5f} and {phi0_b:.5f}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    print(f"log10 slope of the crests: {slope:.3f} per s")
    print(f"lambda = slope x ln 10 = {slope * np.log(10):.3f} per s")

In [ ]:
presets = {
    "12.14: gamma = 1.084, 0 and 1e-5, 7 s": (1.084, 0.0, 1e-5, 7),
    "12.15: gamma = 0.3, 0 and 1 rad, 5 s": (0.3, 0.0, 1.0, 5),
    "Fig. 12.13: gamma = 1.105, 0 and 1e-4, 16 s": (1.105, 0.0, 1e-4, 16),
    "Slides: gamma = 1.5, -pi/2 and -pi/2 + 1e-5, 25 s": (1.5, -np.pi / 2, -np.pi / 2 + 1e-5, 25),
}


@interact(run=presets)
def pick_run(run=presets["12.14: gamma = 1.084, 0 and 1e-5, 7 s"]):
    separation(*run)

For a run of your own, call it directly: `separation(gamma, phi0_a, phi0_b, t_max)`.

## 13. A patch of starting states (Liouville's theorem)

Start 400 pendulums at once, on a $20 \times 20$ grid of $(\phi, \dot\phi)$ values filling a small rectangle around $(-\pi/2, 0)$, $0.01$ rad wide and $0.1$ rad/s tall, and move every one of them with `step_all` from section 11. Each point of the $(\phi, \dot\phi)$ plane moves with velocity $(\dot\phi, \ddot\phi)$, and the divergence of that velocity is $\partial\ddot\phi/\partial\dot\phi = -2\beta$ everywhere, whatever $\gamma$ is. So the area of the patch should shrink as $e^{-2\beta t}$: a factor $e^{-2\beta} = 0.0090$ per drive period.

`blob` plots the 400 states at each time in `times`: on the left a close-up of $t = 0$ and $1$ s (each slid to sit on the origin, so the two shapes can be compared), on the right every time, across the whole attractor. It also prints the area of the four-sided figure through the four corner states at $t = 1$ s, as a fraction of the starting area. (Only at $t = 1$ s: by $t = 2$ s the patch has curled, and its corners no longer outline it.)

**Predict:** look back at your prediction (c) in section 1, then run the cell (a second or two).

In [ ]:
def corner_area(phi, phidot, n):
    """Area of the quadrilateral through the 4 corners of the n x n grid (shoelace formula)."""
    corners = [0, n - 1, n * n - 1, n * (n - 1)]   # in order around the edge
    x = phi[corners]
    y = phidot[corners]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def blob(gamma=1.105, size=(0.01, 0.1), n=20, times=(0, 1, 2, 5, 10)):
    """Evolve an n x n grid of states around (-pi/2, 0); plot it at each time."""
    phi_grid, phidot_grid = np.meshgrid(np.linspace(-size[0] / 2, size[0] / 2, n),
                                        np.linspace(-size[1] / 2, size[1] / 2, n))
    phi = -np.pi / 2 + phi_grid.ravel()
    phidot = phidot_grid.ravel()
    area_start = corner_area(phi, phidot, n)

    fig, (close, wide) = plt.subplots(1, 2, figsize=(12, 5))
    t_now = 0
    for t_show in times:
        n_steps = int(round((t_show - t_now) * steps_per_period))
        phi, phidot = step_all(gamma, phi, phidot, t_now, n_steps)
        t_now = t_show
        if t_show in (0, 1):
            close.plot(phi - phi.mean(), phidot - phidot.mean(), ".", markersize=3,
                       label=f"t = {t_show} s")
        wide.plot(wrap(phi), phidot, ".", markersize=3, label=f"t = {t_show} s")
        if t_show == 1:
            ratio = corner_area(phi, phidot, n) / area_start
            print(f"corner area at t = 1 s / starting area = {ratio:.4f}")
            print(f"e^(-2 beta) = {np.exp(-2 * beta):.4f}")
    close.set_title("close-up: t = 0 and 1 s, each centred on its middle")
    close.set_xlabel("phi - its average (radians)")
    close.set_ylabel("phidot - its average (rad/s)")
    wide.set_title(f"gamma = {gamma}: every time")
    wide.set_xlim(-np.pi, np.pi)
    wide.set_xlabel("phi (radians, wrapped)")
    wide.set_ylabel("phidot (rad/s)")
    for ax in (close, wide):
        ax.grid(alpha=0.3)
        ax.legend()
    plt.show()


blob()